# 01 — Audit exploratoire des données

## Présentation et objectifs

Ce notebook constitue la première étape expérimentale du projet consacré à la généralisation des systèmes de détection d'intrusions entre jeux de données.

L'objectif est de caractériser les deux sources retenues — **CIC-IDS2017** et **UNSW-NB15** — avant leur harmonisation. L'audit porte sur la structure des fichiers, les variables, les labels, la qualité des valeurs et certaines anomalies susceptibles d'influencer les traitements ultérieurs.

Les résultats présentés ici servent à guider les choix de préparation des données. Ils ne constituent pas encore une évaluation des performances des modèles.

### Organisation du notebook

1. Vérification des chemins et exploration initiale des fichiers.
2. Mise en place d'un audit des fichiers CSV par blocs.
3. Analyse structurelle et distribution des labels.
4. Analyse des valeurs manquantes, infinies et des doublons intra-blocs.
5. Inspections complémentaires des colonnes redondantes et des valeurs de ports.
6. Synthèse des observations et transition vers l'harmonisation.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Racine du projet
PROJECT_DIR = Path.cwd().parent

# Répertoires des datasets
CIC_DIR = PROJECT_DIR / "data" / "raw" / "CIC-IDS2017"
UNSW_DIR = PROJECT_DIR / "data" / "raw" / "UNSW-NB15"

print("Projet :", PROJECT_DIR)
print("CIC-IDS2017 existe :", CIC_DIR.exists())
print("UNSW-NB15 existe :", UNSW_DIR.exists())


Projet : D:\Master-SDIA\CYB-1-IDS-Generalization
CIC-IDS2017 existe : True
UNSW-NB15 existe : True


## 1. Exploration initiale de UNSW-NB15

Le jeu de données UNSW-NB15 est accompagné d'un fichier descriptif qui fournit les noms des variables. Les fichiers de données principaux ne possèdent pas de ligne d'en-tête ; les noms issus du fichier descriptif sont donc associés aux colonnes lors de la lecture.

Les cellules suivantes vérifient le dictionnaire des variables, puis affichent un petit échantillon afin d'examiner les valeurs et les types détectés par pandas.

In [2]:
# Lire le fichier descriptif
features_file = UNSW_DIR / "NUSW-NB15_features.csv"

features = pd.read_csv(features_file, encoding="latin1")

# Extraire les noms des caractéristiques
unsw_columns = features["Name"].astype(str).str.strip().tolist()

print("Nombre de colonnes descriptives :", len(unsw_columns))
print("\nNoms des colonnes :")
print(unsw_columns)


Nombre de colonnes descriptives : 49

Noms des colonnes :
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']


In [3]:
# Chemin du premier fichier UNSW-NB15
unsw_file = UNSW_DIR / "UNSW-NB15_1.csv"

# Lire uniquement les 5 premières lignes
# Les CSV UNSW-NB15 n'ont pas d'en-tête
unsw_sample = pd.read_csv(
    unsw_file,
    header=None,
    names=unsw_columns,
    nrows=5
)

print("Dimensions de l'échantillon :", unsw_sample.shape)

print("\nAperçu des données :")
display(unsw_sample)

print("\nTypes des colonnes :")
print(unsw_sample.dtypes)


Dimensions de l'échantillon : (5, 49)

Aperçu des données :


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,0,7,9,1,1,1,1,1,NaN,0



Types des colonnes :
srcip                   str
sport                 int64
dstip                   str
dsport                int64
proto                   str
state                   str
dur                 float64
sbytes                int64
dbytes                int64
sttl                  int64
dttl                  int64
sloss                 int64
dloss                 int64
service                 str
Sload               float64
Dload               float64
Spkts                 int64
Dpkts                 int64
swin                  int64
dwin                  int64
stcpb                 int64
dtcpb                 int64
smeansz               int64
dmeansz               int64
trans_depth           int64
res_bdy_len           int64
Sjit                float64
Djit                float64
Stime                 int64
Ltime                 int64
Sintpkt             float64
Dintpkt             float64
tcprtt                int64
synack                int64
ackdat                int6

### Première observation

Le fichier descriptif fournit **49 noms de variables**. L'aperçu confirme la présence de variables de types différents : adresses IP et protocoles textuels, mesures numériques, catégorie d'attaque et label. Cette diversité justifie de vérifier les types et les valeurs atypiques avant de sélectionner les variables destinées à l'apprentissage.

## 2. Exploration initiale de CIC-IDS2017

Les fichiers CIC-IDS2017 sont recherchés récursivement dans le répertoire du dataset. Un échantillon du premier fichier est ensuite chargé pour examiner sa structure et vérifier si des noms de colonnes sont déjà dupliqués dans cet aperçu.

In [4]:
# Trouver les fichiers CSV CIC
cic_files = sorted(CIC_DIR.rglob("*.csv"))

print("Nombre de fichiers CSV CIC :", len(cic_files))

for file in cic_files:
    print(file.name)

# Lire un échantillon du premier fichier
cic_sample = pd.read_csv(
    cic_files[0],
    nrows=5,
    low_memory=False
)

print("\nDimensions de l'échantillon :", cic_sample.shape)

print("\nAperçu :")
display(cic_sample)

print("\nNombre de colonnes :", len(cic_sample.columns))

# Vérifier les éventuels noms de colonnes dupliqués
duplicated_columns = cic_sample.columns[
    cic_sample.columns.duplicated()
].tolist()

print("Colonnes dupliquées :", duplicated_columns)


Nombre de fichiers CSV CIC : 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv

Dimensions de l'échantillon : (5, 79)

Aperçu :


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6,0,...,20,0,0,0,0,0,0,0,0,BENIGN



Nombre de colonnes : 79
Colonnes dupliquées : []


### Première observation

L'exploration identifie **8 fichiers CSV**. Le premier fichier inspecté contient **79 colonnes**. Aucun nom de colonne strictement dupliqué n'est détecté dans cet échantillon de cinq lignes. Cette vérification initiale ne permet toutefois pas d'exclure la présence de colonnes différentes par leur nom mais contenant les mêmes valeurs ; cette question sera examinée plus loin.

# 01 - Data Audit

## Objectif
Analyser la qualité et la structure des datasets CIC-IDS2017 et UNSW-NB15 avant leur harmonisation.

## Analyses
- Dimensions des fichiers
- Types des variables
- Valeurs manquantes
- Valeurs infinies
- Distribution des labels
- Détection préliminaire des doublons
- Vérification des noms de colonnes

## 3. Configuration de l'audit systématique

Pour traiter les fichiers volumineux sans charger chaque dataset complet en mémoire, l'audit est réalisé par blocs de **100 000 lignes**.

Les résultats sont enregistrés dans `results/tables`. La fonction d'audit collecte notamment le nombre de lignes, les valeurs manquantes, les valeurs infinies numériques, la distribution de la variable `Label` et les doublons détectés à l'intérieur de chaque bloc.

In [5]:
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import re

PROJECT_DIR = Path.cwd().parent

CIC_DIR = PROJECT_DIR / "data" / "raw" / "CIC-IDS2017"
UNSW_DIR = PROJECT_DIR / "data" / "raw" / "UNSW-NB15"

RESULTS_DIR = PROJECT_DIR / "results"
TABLES_DIR = RESULTS_DIR / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 100_000

print("Configuration terminée.")
print("Dossier des résultats :", TABLES_DIR)
print("Taille des blocs :", CHUNK_SIZE)


Configuration terminée.
Dossier des résultats : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables
Taille des blocs : 100000


In [6]:
def audit_csv(file_path, dataset_name, columns=None):
    """
    Audit d'un fichier CSV par blocs.

    columns=None : fichier avec en-tête (CIC-IDS2017)
    columns=list : fichier sans en-tête (UNSW-NB15)
    """

    total_rows = 0
    missing_counts = Counter()
    infinite_counts = Counter()
    label_counts = Counter()

    duplicates_in_chunks = 0
    detected_columns = None

    label_column = "Label"

    reader = pd.read_csv(
        file_path,
        header=0 if columns is None else None,
        names=columns,
        chunksize=CHUNK_SIZE,
        low_memory=False if columns is None else True,
        dtype={"attack_cat": "string"} if columns is not None else None,
        skipinitialspace=True if columns is None else False
    )

    for chunk_number, chunk in enumerate(reader, start=1):

        # Nettoyer uniquement les espaces au début et à la fin
        chunk.columns = chunk.columns.astype(str).str.strip()

        if detected_columns is None:
            detected_columns = chunk.columns.tolist()

        total_rows += len(chunk)

        # Valeurs manquantes
        missing_counts.update(chunk.isna().sum().to_dict())

        # Valeurs infinies dans les variables numériques
        numeric = chunk.select_dtypes(include=[np.number])

        if not numeric.empty:
            inf_counts = np.isinf(numeric).sum()
            infinite_counts.update(inf_counts.to_dict())

        # Doublons détectés à l'intérieur de chaque bloc uniquement
        duplicates_in_chunks += int(chunk.duplicated().sum())

        # Distribution des labels
        if label_column in chunk.columns:
            labels = (
                chunk[label_column]
                .astype("string")
                .str.strip()
                .fillna("<MISSING>")
            )
            label_counts.update(labels.value_counts().to_dict())

    # Détection des noms de colonnes répétés
    # Exemple : Fwd Header Length et Fwd Header Length.1
    logical_columns = [
        re.sub(r"\.\d+$", "", col)
        for col in detected_columns
    ]

    duplicated_names = (
        pd.Series(logical_columns)
        .loc[lambda s: s.duplicated(keep=False)]
        .tolist()
    )

    return {
        "dataset": dataset_name,
        "file": file_path.name,
        "rows": total_rows,
        "columns": len(detected_columns),
        "missing": dict(missing_counts),
        "infinite": dict(infinite_counts),
        "labels": dict(label_counts),
        "duplicates_in_chunks": duplicates_in_chunks,
        "duplicated_column_names": sorted(set(duplicated_names)),
        "column_names": detected_columns
    }

print("Fonction d'audit prête.")


Fonction d'audit prête.


## 4. Audit des fichiers UNSW-NB15

Les quatre fichiers principaux `UNSW-NB15_1.csv` à `UNSW-NB15_4.csv` sont parcourus par blocs. Le fichier descriptif est utilisé pour associer les noms aux colonnes.

Une normalisation ciblée du nom `ct_src_ ltm` est appliquée pour supprimer l'espace parasite dans le nom de variable. Cette étape concerne le nom de colonne et ne modifie pas les valeurs du dataset.

In [7]:
features_file = UNSW_DIR / "NUSW-NB15_features.csv"

features = pd.read_csv(features_file, encoding="latin1")
unsw_columns = features["Name"].astype(str).str.strip().tolist()

# Normalisation documentée d'un nom de colonne
unsw_columns = [
    col.replace("ct_src_ ltm", "ct_src_ltm")
    for col in unsw_columns
]

unsw_files = []

for file in UNSW_DIR.rglob("UNSW-NB15_*.csv"):
    if re.fullmatch(r"UNSW-NB15_[1-4]\.csv", file.name):
        unsw_files.append(file)

unsw_files = sorted(unsw_files)

unsw_results = []

for file in unsw_files:
    print("Audit en cours :", file.name)

    result = audit_csv(
        file_path=file,
        dataset_name="UNSW-NB15",
        columns=unsw_columns
    )

    unsw_results.append(result)

print("\nAudit UNSW-NB15 terminé.")


Audit en cours : UNSW-NB15_1.csv


C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (1: sport, 3: dsport) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):
C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (1: sport, 3: dsport) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):


Audit en cours : UNSW-NB15_2.csv


C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (3: dsport, 39: ct_ftp_cmd) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):
C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (3: dsport) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):
C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (3: dsport) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):
C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (3: dsport) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):


Audit en cours : UNSW-NB15_3.csv


C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (3: dsport) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):
C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (3: dsport) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):
C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (3: dsport) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):
C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (3: dsport) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):
C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarni

Audit en cours : UNSW-NB15_4.csv


C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (3: dsport) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):
C:\Users\eljir\AppData\Local\Temp\ipykernel_876\3050566655.py:29: DtypeWarning: Columns (3: dsport) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):



Audit UNSW-NB15 terminé.


## 5. Audit des fichiers CIC-IDS2017

Les huit fichiers CSV CIC-IDS2017 sont à leur tour parcourus par blocs. Contrairement aux fichiers UNSW, ils possèdent leurs propres en-têtes. Les espaces superflus au début et à la fin des noms de colonnes sont nettoyés pendant la lecture.

In [8]:
cic_files = sorted(CIC_DIR.rglob("*.csv"))

cic_results = []

for file in cic_files:
    print("Audit en cours :", file.name)

    result = audit_csv(
        file_path=file,
        dataset_name="CIC-IDS2017"
    )

    cic_results.append(result)

print("\nAudit CIC-IDS2017 terminé.")


Audit en cours : Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Audit en cours : Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Audit en cours : Friday-WorkingHours-Morning.pcap_ISCX.csv
Audit en cours : Monday-WorkingHours.pcap_ISCX.csv
Audit en cours : Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Audit en cours : Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Audit en cours : Tuesday-WorkingHours.pcap_ISCX.csv
Audit en cours : Wednesday-workingHours.pcap_ISCX.csv

Audit CIC-IDS2017 terminé.


## 6. Synthèse structurelle des fichiers

Le tableau ci-dessous rassemble les résultats de l'audit fichier par fichier : nombre de lignes, nombre de colonnes, doublons détectés à l'intérieur des blocs et noms de colonnes potentiellement redondants.

**Précaution d'interprétation :** le compteur `Doublons intra-blocs` ne mesure pas les doublons globaux sur l'ensemble du fichier. La fonction compare les lignes au sein de chaque bloc indépendamment ; des doublons situés dans deux blocs différents peuvent donc ne pas être comptabilisés.

In [9]:
all_results = unsw_results + cic_results

summary = pd.DataFrame([
    {
        "Dataset": r["dataset"],
        "Fichier": r["file"],
        "Nombre de lignes": r["rows"],
        "Nombre de colonnes": r["columns"],
        "Doublons intra-blocs": r["duplicates_in_chunks"],
        "Colonnes potentiellement dupliquées": str(
            r["duplicated_column_names"]
        )
    }
    for r in all_results
])

display(summary)

# Enregistrement du tableau
summary.to_csv(
    TABLES_DIR / "dataset_dimensions.csv",
    index=False,
    encoding="utf-8"
)

print("Tableau enregistré :", TABLES_DIR / "dataset_dimensions.csv")


,Dataset,Fichier,Nombre de lignes,Nombre de colonnes,Doublons intra-blocs,Colonnes potentiellement dupliquées
0,UNSW-NB15,UNSW-NB15_1.csv,700001,49,59209,[]
1,UNSW-NB15,UNSW-NB15_2.csv,700001,49,63244,[]
2,UNSW-NB15,UNSW-NB15_3.csv,700001,49,224836,[]
3,UNSW-NB15,UNSW-NB15_4.csv,440044,49,133311,[]
4,CIC-IDS2017,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,79,1870,['Fwd Header Length']
5,CIC-IDS2017,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,79,47775,['Fwd Header Length']
6,CIC-IDS2017,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,79,5404,['Fwd Header Length']
7,CIC-IDS2017,Monday-WorkingHours.pcap_ISCX.csv,529918,79,17480,['Fwd Header Length']
8,CIC-IDS2017,Thursday-WorkingHours-Afternoon-Infilteration....,288602,79,28104,['Fwd Header Length']
9,CIC-IDS2017,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,79,4829,['Fwd Header Length']


Tableau enregistré : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\dataset_dimensions.csv


### Lecture des résultats

La synthèse permet de comparer les tailles des fichiers et de repérer les colonnes qui méritent une inspection complémentaire. Les nombres de doublons doivent être interprétés comme des doublons intra-blocs, et non comme une mesure exhaustive de l'unicité des lignes.

## 7. Distribution des labels

Cette analyse compte les occurrences de chaque valeur de `Label` dans chaque fichier. Elle permet de décrire la représentation des classes normales et des classes d'attaque, et d'identifier les différences de distribution entre fichiers.

La distribution est conservée dans `results/tables/label_distribution.csv` pour être réutilisée dans les étapes d'analyse.

In [10]:
label_rows = []

for result in all_results:
    for label, count in result["labels"].items():
        label_rows.append({
            "Dataset": result["dataset"],
            "Fichier": result["file"],
            "Label": label,
            "Nombre": count
        })

labels_df = pd.DataFrame(label_rows)

display(labels_df)

labels_df.to_csv(
    TABLES_DIR / "label_distribution.csv",
    index=False,
    encoding="utf-8"
)

print("Distribution des labels enregistrée.")


,Dataset,Fichier,Label,Nombre
0,UNSW-NB15,UNSW-NB15_1.csv,0,677786
1,UNSW-NB15,UNSW-NB15_1.csv,1,22215
2,UNSW-NB15,UNSW-NB15_2.csv,0,647252
3,UNSW-NB15,UNSW-NB15_2.csv,1,52749
4,UNSW-NB15,UNSW-NB15_3.csv,0,542576
5,UNSW-NB15,UNSW-NB15_3.csv,1,157425
6,UNSW-NB15,UNSW-NB15_4.csv,0,351150
7,UNSW-NB15,UNSW-NB15_4.csv,1,88894
8,CIC-IDS2017,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,DDoS,128027
9,CIC-IDS2017,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,BENIGN,97718


Distribution des labels enregistrée.


### Interprétation

Les effectifs doivent être examinés par dataset et par fichier : une distribution globale peut masquer des variations importantes entre les journées ou scénarios de capture. Le déséquilibre des classes constitue un élément à prendre en compte lors du choix des métriques d'évaluation, notamment la précision, le rappel et l'Average Precision.

## 8. Audit des valeurs manquantes et infinies

Les résultats de la fonction d'audit sont regroupés par dataset, fichier et variable. Cette étape identifie les colonnes présentant des valeurs manquantes (`NaN`) ou infinies dans les variables numériques.

L'objectif est de repérer les variables nécessitant une attention particulière avant l'harmonisation et l'entraînement des modèles.

In [11]:
quality_rows = []

for result in all_results:
    for column, count in result["missing"].items():
        quality_rows.append({
            "Dataset": result["dataset"],
            "Fichier": result["file"],
            "Variable": column,
            "Valeurs manquantes": count,
            "Valeurs infinies": result["infinite"].get(column, 0)
        })

quality_df = pd.DataFrame(quality_rows)

display(
    quality_df[
        (quality_df["Valeurs manquantes"] > 0) |
        (quality_df["Valeurs infinies"] > 0)
    ].sort_values(
        ["Dataset", "Valeurs manquantes"],
        ascending=[True, False]
    )
)

quality_df.to_csv(
    TABLES_DIR / "data_quality.csv",
    index=False,
    encoding="utf-8"
)

print("Rapport qualité enregistré.")


,Dataset,Fichier,Variable,Valeurs manquantes,Valeurs infinies
763,CIC-IDS2017,Wednesday-workingHours.pcap_ISCX.csv,Flow Bytes/s,1008,289
684,CIC-IDS2017,Tuesday-WorkingHours.pcap_ISCX.csv,Flow Bytes/s,201,63
447,CIC-IDS2017,Monday-WorkingHours.pcap_ISCX.csv,Flow Bytes/s,64,373
368,CIC-IDS2017,Friday-WorkingHours-Morning.pcap_ISCX.csv,Flow Bytes/s,28,94
605,CIC-IDS2017,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Flow Bytes/s,20,115
526,CIC-IDS2017,Thursday-WorkingHours-Afternoon-Infilteration....,Flow Bytes/s,18,189
289,CIC-IDS2017,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Flow Bytes/s,15,356
210,CIC-IDS2017,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow Bytes/s,4,30
211,CIC-IDS2017,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Flow Packets/s,0,34
290,CIC-IDS2017,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Flow Packets/s,0,371


Rapport qualité enregistré.


### Interprétation

Les anomalies observées doivent être traitées en fonction de leur nature et de leur fréquence. Leur présence ne conduit pas automatiquement à supprimer une variable ou une ligne : la décision doit être prise après examen de la signification de la variable et de son utilité pour le protocole expérimental.

## 9. Inspections complémentaires des anomalies

Cette partie regroupe des vérifications ciblées à partir des premiers résultats : inspection de colonnes susceptibles d'être redondantes, examen de certaines valeurs non numériques et vérification d'anomalies détectées dans les données.

Les résultats de cette étape servent à formuler des décisions de préparation documentées plutôt qu'à appliquer automatiquement des corrections.

In [12]:
# ============================================================
# Inspection complémentaire des anomalies
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. Inspection des colonnes dupliquées dans CIC-IDS2017
# ------------------------------------------------------------

duplicate_column_results = []

for file_path in cic_files:
    print(f"Inspection : {file_path.name}")

    # Lecture d'un échantillon pour identifier les colonnes
    sample = pd.read_csv(
        file_path,
        nrows=10000,
        low_memory=False,
        skipinitialspace=True
    )

    sample.columns = sample.columns.str.strip()

    first_col = "Fwd Header Length"
    second_col = "Fwd Header Length.1"

    if first_col in sample.columns and second_col in sample.columns:
        col1 = pd.to_numeric(sample[first_col], errors="coerce")
        col2 = pd.to_numeric(sample[second_col], errors="coerce")

        equal = col1.eq(col2) | (col1.isna() & col2.isna())

        duplicate_column_results.append({
            "Fichier": file_path.name,
            "Colonnes_comparees": f"{first_col} / {second_col}",
            "Valeurs_identiques": int(equal.sum()),
            "Valeurs_differentes": int((~equal).sum()),
            "Echantillon": len(sample)
        })

duplicate_columns_df = pd.DataFrame(duplicate_column_results)

display(duplicate_columns_df)


# ------------------------------------------------------------
# 2. Inspection des valeurs non numériques dans UNSW
# ------------------------------------------------------------

unsw_file = unsw_files[0]

unsw_sample = pd.read_csv(
    unsw_file,
    header=None,
    names=unsw_columns,
    nrows=100000,
    dtype={
        "sport": "string",
        "dsport": "string",
        "ct_ftp_cmd": "string"
    }
)

for column in ["sport", "dsport", "ct_ftp_cmd"]:
    values = unsw_sample[column].dropna()

    numeric_values = pd.to_numeric(
        values,
        errors="coerce"
    )

    invalid_values = values[numeric_values.isna()]

    print(f"\nVariable : {column}")
    print("Nombre de valeurs non numériques :", len(invalid_values))
    print("Exemples :", invalid_values.unique()[:15])


# ------------------------------------------------------------
# 3. Vérification des proportions de doublons intra-blocs
# ------------------------------------------------------------

duplicates_report = summary.copy()

duplicates_report["Taux_doublons_intra_blocs_pct"] = (
    duplicates_report["Doublons intra-blocs"]
    / duplicates_report["Nombre de lignes"]
    * 100
)

display(
    duplicates_report[
        [
            "Dataset",
            "Fichier",
            "Nombre de lignes",
            "Doublons intra-blocs",
            "Taux_doublons_intra_blocs_pct"
        ]
    ]
)


Inspection : Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Inspection : Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Inspection : Friday-WorkingHours-Morning.pcap_ISCX.csv
Inspection : Monday-WorkingHours.pcap_ISCX.csv
Inspection : Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Inspection : Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Inspection : Tuesday-WorkingHours.pcap_ISCX.csv
Inspection : Wednesday-workingHours.pcap_ISCX.csv


,Fichier,Colonnes_comparees,Valeurs_identiques,Valeurs_differentes,Echantillon
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Fwd Header Length / Fwd Header Length.1,10000,0,10000
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Fwd Header Length / Fwd Header Length.1,10000,0,10000
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,Fwd Header Length / Fwd Header Length.1,10000,0,10000
3,Monday-WorkingHours.pcap_ISCX.csv,Fwd Header Length / Fwd Header Length.1,10000,0,10000
4,Thursday-WorkingHours-Afternoon-Infilteration....,Fwd Header Length / Fwd Header Length.1,10000,0,10000
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Fwd Header Length / Fwd Header Length.1,10000,0,10000
6,Tuesday-WorkingHours.pcap_ISCX.csv,Fwd Header Length / Fwd Header Length.1,10000,0,10000
7,Wednesday-workingHours.pcap_ISCX.csv,Fwd Header Length / Fwd Header Length.1,10000,0,10000



Variable : sport
Nombre de valeurs non numériques : 5
Exemples : <StringArray>
['0x000b', '0x000c', '-']
Length: 3, dtype: string

Variable : dsport
Nombre de valeurs non numériques : 61
Exemples : <StringArray>
['0xc0a8', '-', '0x20205321']
Length: 3, dtype: string

Variable : ct_ftp_cmd
Nombre de valeurs non numériques : 0
Exemples : <StringArray>
[]
Length: 0, dtype: string


,Dataset,Fichier,Nombre de lignes,Doublons intra-blocs,Taux_doublons_intra_blocs_pct
0,UNSW-NB15,UNSW-NB15_1.csv,700001,59209,8.458416
1,UNSW-NB15,UNSW-NB15_2.csv,700001,63244,9.034844
2,UNSW-NB15,UNSW-NB15_3.csv,700001,224836,32.119383
3,UNSW-NB15,UNSW-NB15_4.csv,440044,133311,30.294925
4,CIC-IDS2017,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,1870,0.828368
5,CIC-IDS2017,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,47775,16.677314
6,CIC-IDS2017,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,5404,2.828831
7,CIC-IDS2017,Monday-WorkingHours.pcap_ISCX.csv,529918,17480,3.298624
8,CIC-IDS2017,Thursday-WorkingHours-Afternoon-Infilteration....,288602,28104,9.737978
9,CIC-IDS2017,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,4829,2.834486


## 10. Vérification complète de deux colonnes CIC potentiellement redondantes

L'inspection initiale a mis en évidence deux variables proches par leur nom : `Fwd Header Length` et `Fwd Header Length.1`. Une comparaison est réalisée sur l'ensemble des lignes des huit fichiers, par blocs de 100 000 lignes.

La comparaison considère également comme égales deux valeurs manquantes situées sur la même ligne. Le tableau obtenu indique, pour chaque fichier, le nombre de lignes comparées, les valeurs identiques et les valeurs différentes.

In [13]:
import pandas as pd

full_comparison = []

for file_path in cic_files:

    print(f"Comparaison complète : {file_path.name}")

    identical = 0
    different = 0
    total = 0

    reader = pd.read_csv(
        file_path,
        usecols=["Fwd Header Length", "Fwd Header Length.1"],
        chunksize=100_000,
        low_memory=False,
        skipinitialspace=True
    )

    for chunk in reader:
        col1 = chunk["Fwd Header Length"]
        col2 = chunk["Fwd Header Length.1"]

        equal = col1.eq(col2) | (col1.isna() & col2.isna())

        identical += int(equal.sum())
        different += int((~equal).sum())
        total += len(chunk)

    full_comparison.append({
        "Fichier": file_path.name,
        "Nombre_lignes": total,
        "Valeurs_identiques": identical,
        "Valeurs_differentes": different
    })

full_comparison_df = pd.DataFrame(full_comparison)

display(full_comparison_df)


Comparaison complète : Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Comparaison complète : Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Comparaison complète : Friday-WorkingHours-Morning.pcap_ISCX.csv
Comparaison complète : Monday-WorkingHours.pcap_ISCX.csv
Comparaison complète : Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Comparaison complète : Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Comparaison complète : Tuesday-WorkingHours.pcap_ISCX.csv
Comparaison complète : Wednesday-workingHours.pcap_ISCX.csv


,Fichier,Nombre_lignes,Valeurs_identiques,Valeurs_differentes
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,225745,0
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,286467,0
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,191033,0
3,Monday-WorkingHours.pcap_ISCX.csv,529918,529918,0
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,288602,0
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,170366,0
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,445909,0
7,Wednesday-workingHours.pcap_ISCX.csv,692703,692703,0


### Résultat de la comparaison

Les résultats affichés montrent que les deux colonnes sont identiques pour toutes les lignes parcourues dans chacun des huit fichiers. Cette vérification complète est plus probante que la comparaison initiale sur un petit échantillon.

## 11. Inspection des variables de ports dans UNSW-NB15

Les variables `sport` et `dsport` sont lues comme chaînes de caractères afin de ne pas perdre les valeurs qui ne suivent pas le format numérique attendu. Les valeurs pouvant être converties en nombres sont distinguées de celles qui ne le peuvent pas.

Cette inspection permet notamment de repérer des représentations hexadécimales et le caractère `-`. Ces observations sont importantes si ces variables sont envisagées pour une analyse ultérieure.

In [14]:
port_values = {
    "sport": {},
    "dsport": {}
}

for file_path in unsw_files:

    print(f"Inspection des ports : {file_path.name}")

    reader = pd.read_csv(
        file_path,
        header=None,
        names=unsw_columns,
        usecols=[1, 3],
        chunksize=100_000,
        dtype={
            "sport": "string",
            "dsport": "string"
        }
    )

    for chunk in reader:
        for col in ["sport", "dsport"]:

            values = chunk[col].dropna().astype(str)

            invalid = values[
                pd.to_numeric(values, errors="coerce").isna()
            ]

            counts = invalid.value_counts()

            for value, count in counts.items():
                port_values[col][value] = (
                    port_values[col].get(value, 0) + int(count)
                )

for col, values in port_values.items():
    print(f"\nValeurs non numériques de {col} :")

    display(
        pd.DataFrame(
            values.items(),
            columns=["Valeur", "Nombre"]
        ).sort_values("Nombre", ascending=False)
    )


Inspection des ports : UNSW-NB15_1.csv
Inspection des ports : UNSW-NB15_2.csv
Inspection des ports : UNSW-NB15_3.csv
Inspection des ports : UNSW-NB15_4.csv

Valeurs non numériques de sport :


,Valeur,Nombre
1,0x000c,4
0,0x000b,2
2,-,2



Valeurs non numériques de dsport :


,Valeur,Nombre
3,0xcc09,241
0,0xc0a8,54
1,-,7
2,0x20205321,2


### Interprétation

La présence de valeurs telles que `0x...` et `-` confirme que les colonnes de ports ne sont pas uniformément représentées comme des entiers décimaux dans les fichiers. Il ne faut donc pas effectuer une conversion numérique aveugle. Dans le cadre du schéma initial partagé entre les deux datasets, les ports ne sont de toute façon pas retenus parmi les cinq variables communes sélectionnées dans le notebook d'harmonisation.

## 12. Vérifications complémentaires sur le dernier fichier CIC

Les cellules suivantes reprennent le dernier fichier CIC dans la synthèse afin de vérifier explicitement ses dimensions et le résultat de la comparaison des deux colonnes `Fwd Header Length`. Elles constituent un contrôle complémentaire ciblé sur le fichier `Wednesday-workingHours.pcap_ISCX.csv`.

In [15]:
print("Colonnes de summary :")
print(summary.columns.tolist())

print("\nDernier fichier audité :")
display(
    summary[
        summary["Fichier"].str.contains(
            "Wednesday-workingHours",
            na=False
        )
    ]
)


Colonnes de summary :
['Dataset', 'Fichier', 'Nombre de lignes', 'Nombre de colonnes', 'Doublons intra-blocs', 'Colonnes potentiellement dupliquées']

Dernier fichier audité :


,Dataset,Fichier,Nombre de lignes,Nombre de colonnes,Doublons intra-blocs,Colonnes potentiellement dupliquées
11,CIC-IDS2017,Wednesday-workingHours.pcap_ISCX.csv,692703,79,71964,['Fwd Header Length']


In [16]:
# Vérification du dernier fichier CIC

wednesday_file = next(
    f for f in cic_files
    if f.name == "Wednesday-workingHours.pcap_ISCX.csv"
)

print("Fichier :", wednesday_file.name)

reader = pd.read_csv(
    wednesday_file,
    usecols=["Fwd Header Length", "Fwd Header Length.1"],
    chunksize=100_000,
    low_memory=False,
    skipinitialspace=True
)

identical = 0
different = 0
total = 0

for chunk in reader:
    col1 = chunk["Fwd Header Length"]
    col2 = chunk["Fwd Header Length.1"]

    equal = col1.eq(col2) | (col1.isna() & col2.isna())

    identical += int(equal.sum())
    different += int((~equal).sum())
    total += len(chunk)

print("Nombre total de lignes :", total)
print("Valeurs identiques :", identical)
print("Valeurs différentes :", different)

Fichier : Wednesday-workingHours.pcap_ISCX.csv
Nombre total de lignes : 692703
Valeurs identiques : 692703
Valeurs différentes : 0


In [17]:
print(
    full_comparison_df[
        full_comparison_df["Fichier"].str.contains(
            "Wednesday-workingHours",
            na=False
        )
    ].to_string(index=False)
)

                             Fichier  Nombre_lignes  Valeurs_identiques  Valeurs_differentes
Wednesday-workingHours.pcap_ISCX.csv         692703              692703                    0


## 13. Enregistrement des résultats de comparaison

Le tableau de comparaison complète est exporté au format CSV dans `results/tables/cic_duplicate_columns_verification.csv`. Le total des lignes vérifiées et le nombre total de différences sont également affichés.

In [18]:
# Résultat de la vérification des colonnes redondantes

full_comparison_df.to_csv(
    "../results/tables/cic_duplicate_columns_verification.csv",
    index=False
)

print("Vérification enregistrée.")
print(
    "Total des lignes vérifiées :",
    full_comparison_df["Nombre_lignes"].sum()
)

print(
    "Total des différences :",
    full_comparison_df["Valeurs_differentes"].sum()
)

Vérification enregistrée.
Total des lignes vérifiées : 2830743
Total des différences : 0


## 14. Synthèse et conclusion de l'audit

Cet audit a permis de caractériser les fichiers CIC-IDS2017 et UNSW-NB15 avant leur harmonisation. Il a notamment mis en évidence :

- des structures et des nombres de variables différents entre les deux sources ;
- des distributions de labels à examiner par fichier et par dataset ;
- des valeurs manquantes ou infinies à prendre en compte dans la préparation ;
- des valeurs non numériques dans les variables `sport` et `dsport` de UNSW-NB15 ;
- une redondance confirmée entre `Fwd Header Length` et `Fwd Header Length.1` sur l'ensemble des lignes CIC vérifiées ;
- une mesure des doublons limitée aux comparaisons effectuées à l'intérieur de chaque bloc.

### Limites méthodologiques

- Le comptage des doublons de la fonction d'audit est intra-bloc ; il ne constitue pas un décompte global exhaustif.
- Les premières inspections sur cinq ou quelques milliers de lignes sont exploratoires ; elles sont distinguées des vérifications complètes réalisées ensuite.
- La présence d'une anomalie ne suffit pas, à elle seule, à justifier la suppression d'une variable ou d'une observation. Les décisions de préparation doivent être explicitées dans l'étape suivante.

### Transition vers le notebook 02

Le notebook suivant porte sur l'harmonisation des deux datasets : définition d'un schéma de variables communes, normalisation des unités lorsque cela est nécessaire, construction d'échantillons comparables et préparation des données pour les expérimentations de généralisation.